In [2]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, Input
from tensorflow.keras.layers import ConvLSTM2D, BatchNormalization, Conv3D, Dense, GlobalAveragePooling1D
from tensorflow.keras.layers import Embedding, TextVectorization, Concatenate, RepeatVector, Reshape
from tensorflow.keras.models import Model
import pandas as pd
import cv2

2025-04-05 06:35:40.431621: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-05 06:35:40.443221: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1743834940.454708      44 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1743834940.458092      44 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1743834940.466954      44 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [3]:
from transformers import TFCLIPModel

/home/mpstme-bhavya/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
VIDEO_DIR = r"/data/mpstme-bhavya/small_videos"  # Folder containing .npy video arrays
PROMPT_CSV = r"/data/mpstme-bhavya/small_dataset.csv"  # CSV with columns: filename, prompt

In [5]:
FRAME_COUNT = 32  # Extract 32 frames per video
IMAGE_SIZE = 128  # Resize frames to 128x128

In [6]:
prompt_df = pd.read_csv(PROMPT_CSV)

In [7]:
def load_video_frames(video_path, frame_count=FRAME_COUNT, size=(IMAGE_SIZE, IMAGE_SIZE)):
    cap = cv2.VideoCapture(video_path)
    frames = []
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Select frame indices evenly spaced
    frame_indices = np.linspace(0, total_frames - 1, frame_count, dtype=int)

    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret:
            break  # Stop if frame not read
        frame = cv2.resize(frame, size)
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) / 255.0  # Normalize
        frames.append(frame)

    cap.release()

    # Pad if less than required frames
    while len(frames) < frame_count:
        frames.append(np.zeros((size[0], size[1], 3)))

    return np.array(frames)

In [8]:
from transformers import TFCLIPModel, CLIPProcessor

clip_model = TFCLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")


W0000 00:00:1743834944.841487      44 gpu_device.cc:2341] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...
All model checkpoint layers were used when initializing TFCLIPModel.

All the layers of TFCLIPModel were initialized from the model checkpoint at openai/clip-vit-base-patch32.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFCLIPModel for predictions without further training.


In [9]:
def encode_text_clip(texts):
    inputs = clip_processor(text=texts, return_tensors="tf", padding=True, truncation=True)
    text_features = clip_model.get_text_features(**inputs)
    return text_features.numpy() 

In [10]:
from tqdm import tqdm

In [11]:
video_data = []
text_embeddings = []

# Loop through dataset to process videos and text prompts
for _, row in tqdm(prompt_df.iterrows(), total=len(prompt_df)):
    video_path = os.path.join(VIDEO_DIR, row["video_file"])
    frames = load_video_frames(video_path)  # You should have this function
    embedding = encode_text_clip([row["caption"]])[0]  # Single text prompt

    video_data.append(frames)
    text_embeddings.append(embedding)

# Convert lists to numpy arrays
video_data = np.array(video_data)  # Shape: (N, 32, 128, 128, 3)
text_embeddings = np.array(text_embeddings)  # Shape: (N, 512)

print("Video Data Shape:", video_data.shape)
print("Text Embeddings Shape:", text_embeddings.shape)

100%|██████████| 200/200 [02:16<00:00,  1.47it/s]


Video Data Shape: (200, 32, 128, 128, 3)
Text Embeddings Shape: (200, 512)


In [12]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [13]:
def build_encoder():
    # Inputs
    video_input = keras.Input(shape=(32, 128, 128, 3), name="video_input")
    text_input = keras.Input(shape=(512,), name="text_embedding")  # Add text input

    # CNN Feature Extraction for video
    x = layers.TimeDistributed(layers.Conv2D(64, (3, 3), activation="relu"))(video_input)
    x = layers.TimeDistributed(layers.MaxPooling2D((2, 2)))(x)
    x = layers.TimeDistributed(layers.Conv2D(128, (3, 3), activation="relu"))(x)
    x = layers.TimeDistributed(layers.MaxPooling2D((2, 2)))(x)

    # Flatten spatial features
    x = layers.TimeDistributed(layers.Flatten())(x)

    # LSTM over time dimension (frames)
    x = layers.LSTM(512, return_sequences=True)(x)
    x = layers.LSTM(512, return_sequences=False, name="video_lstm_output")(x)

    # Combine with text embedding
    combined = layers.Concatenate()([x, text_input])  # Shape: (B, 1024)

    # Optional: Dense layer to compress to latent dimension
    z = layers.Dense(512, activation="relu", name="encoder_output")(combined)

    # Define Model
    encoder_model = keras.Model(inputs=[video_input, text_input], outputs=z, name="video_encoder")
    return encoder_model


In [14]:
encoder = build_encoder()
encoder.summary()

Model: "video_encoder"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ video_input         │ (None, 32, 128,   │          0 │ -                 │
│ (InputLayer)        │ 128, 3)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed    │ (None, 32, 126,   │      1,792 │ video_input[0][0] │
│ (TimeDistributed)   │ 126, 64)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_1  │ (None, 32, 63,    │          0 │ time_distributed… │
│ (TimeDistributed)   │ 63, 64)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_2  │ (None, 32, 61,    │     73,856 │ time_distributed… │
│ (TimeDistributed)   │ 61, 128)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_3  │ (None, 32, 30,    │          0 │ time_distributed… │
│ (TimeDistributed)   │ 30, 128)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_4  │ (None, 32,        │          0 │ time_distributed… │
│ (TimeDistributed)   │ 115200)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 32, 512)   │ 236,980,2… │ time_distributed… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ video_lstm_output   │ (None, 512)       │  2,099,200 │ lstm[0][0]        │
│ (LSTM)              │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text_embedding      │ (None, 512)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 1024)      │          0 │ video_lstm_outpu… │
│ (Concatenate)       │                   │            │ text_embedding[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_output      │ (None, 512)       │    524,800 │ concatenate[0][0] │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 239,679,872 (914.31 MB)

 Trainable params: 239,679,872 (914.31 MB)

 Non-trainable params: 0 (0.00 B)

In [15]:
def build_decoder():
    decoder_input = keras.Input(shape=(512,), name="decoder_input")

    # Repeat latent vector across 32 frames
    x = layers.RepeatVector(32)(decoder_input)  # (B, 32, 512)

    # Temporal modeling
    x = layers.LSTM(512, return_sequences=True)(x)

    # Project each time step to 128x128x3 frame
    x = layers.TimeDistributed(layers.Dense(128 * 128 * 3, activation="relu"))(x)
    x = layers.TimeDistributed(layers.Reshape((128, 128, 3)))(x)  # (B, 32, 128, 128, 3)

    decoder_output = x  # video frames

    decoder_model = keras.Model(inputs=decoder_input, outputs=decoder_output, name="video_decoder")
    return decoder_model


In [16]:
decoder = build_decoder()
decoder.summary()

Model: "video_decoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ decoder_input (InputLayer)      │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector (RepeatVector)    │ (None, 32, 512)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32, 512)        │     2,099,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_5              │ (None, 32, 49152)      │    25,214,976 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_6              │ (None, 32, 128, 128,   │             0 │
│ (TimeDistributed)               │ 3)                     │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 27,314,176 (104.20 MB)

 Trainable params: 27,314,176 (104.20 MB)

 Non-trainable params: 0 (0.00 B)

In [17]:
video_input = keras.Input(shape=(32, 128, 128, 3), name="video_input")
text_embedding_input = keras.Input(shape=(512,), name="text_embedding")  # Add this

# Pass both inputs to encoder
encoded_video = encoder([video_input, text_embedding_input])

decoded_text_embedding = decoder(encoded_video)

# Make sure to also provide both inputs in model creation
video_to_text_model = keras.Model(
    inputs=[video_input, text_embedding_input],
    outputs=decoded_text_embedding,
    name="video_to_text_model"
)


In [18]:
video_to_text_model.summary()

Model: "video_to_text_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ video_input         │ (None, 32, 128,   │          0 │ -                 │
│ (InputLayer)        │ 128, 3)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text_embedding      │ (None, 512)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ video_encoder       │ (None, 512)       │ 239,679,8… │ video_input[0][0… │
│ (Functional)        │                   │            │ text_embedding[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ video_decoder       │ (None, 32, 128,   │ 27,314,176 │ video_encoder[0]… │
│ (Functional)        │ 128, 3)           │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 266,994,048 (1018.50 MB)

 Trainable params: 266,994,048 (1018.50 MB)

 Non-trainable params: 0 (0.00 B)

In [19]:
class TimestepEmbedding(layers.Layer):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def call(self, t):
        half_dim = self.dim // 2
        emb = tf.cast(t, tf.float32)[:, None]
        freqs = tf.exp(
            -tf.math.log(10000.0) * tf.range(half_dim, dtype=tf.float32) / half_dim
        )
        emb = emb * freqs
        emb = tf.concat([tf.sin(emb), tf.cos(emb)], axis=1)
        return emb

In [20]:
def diffusion_unet(input_shape=(128, 128, 3), embedding_dim=128, filters=64):
    """
    Diffusion U-Net model using Functional API with sinusoidal timestep embedding.
    
    Args:
        input_shape: Shape of input images (Height, Width, Channels).
        embedding_dim: Dimension of the time embedding.
        filters: Base number of filters for convolution layers.

    Returns:
        A Keras Model for diffusion-based noise estimation.
    """
    # Inputs
    x_input = layers.Input(shape=input_shape, name="image_input")     # (B, H, W, C)
    t_input = layers.Input(shape=(), dtype=tf.int32, name="timestep") # (B,)

    # Timestep embedding
    t_embed = TimestepEmbedding(embedding_dim)(t_input)               # (B, embedding_dim)
    t_embed = layers.Dense(filters, activation="relu")(t_embed)
    t_embed = layers.Dense(filters, activation="relu")(t_embed)
    
    # Project and reshape to (B, 1, 1, C) to match spatial dimensions
    t_proj = layers.Dense(input_shape[-1], activation="linear")(t_embed)  # Match channels
    t_proj = layers.Reshape((1, 1, input_shape[-1]))(t_proj)              # (B, 1, 1, C)

    # Inject timestep into input image
    x = layers.Add()([x_input, t_proj])  # (B, H, W, C)

    # Encoder
    x1 = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
    p1 = layers.MaxPooling2D(2)(x1)

    x2 = layers.Conv2D(filters * 2, 3, padding="same", activation="relu")(p1)
    p2 = layers.MaxPooling2D(2)(x2)

    x3 = layers.Conv2D(filters * 4, 3, padding="same", activation="relu")(p2)
    p3 = layers.MaxPooling2D(2)(x3)

    # Bottleneck
    x = layers.Conv2D(filters * 8, 3, padding="same", activation="relu")(p3)

    # Decoder
    x = layers.Conv2DTranspose(filters * 4, 3, strides=2, padding="same", activation="relu")(x)
    x = layers.Concatenate()([x3, x])
    x = layers.Conv2D(filters * 4, 3, padding="same", activation="relu")(x)

    x = layers.Conv2DTranspose(filters * 2, 3, strides=2, padding="same", activation="relu")(x)
    x = layers.Concatenate()([x2, x])
    x = layers.Conv2D(filters * 2, 3, padding="same", activation="relu")(x)

    x = layers.Conv2DTranspose(filters, 3, strides=2, padding="same", activation="relu")(x)
    x = layers.Concatenate()([x1, x])
    x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)

    # Output layer
    output = layers.Conv2D(3, 3, padding="same", activation="sigmoid")(x)

    return Model(inputs=[x_input, t_input], outputs=output, name="DiffusionUNet")

In [21]:
diffusion_model = diffusion_unet()
diffusion_model.summary()


Model: "DiffusionUNet"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ timestep            │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ timestep_embedding  │ (None, 128)       │          0 │ timestep[0][0]    │
│ (TimestepEmbedding) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64)        │      8,256 │ timestep_embeddi… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 64)        │      4,160 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 3)         │        195 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ image_input         │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_1 (Reshape) │ (None, 1, 1, 3)   │          0 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 128, 128,  │          0 │ image_input[0][0… │
│                     │ 3)                │            │ reshape_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 128, 128,  │      1,792 │ add[0][0]         │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 64, 64,    │          0 │ conv2d_2[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 64, 64,    │     73,856 │ max_pooling2d_2[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_3     │ (None, 32, 32,    │          0 │ conv2d_3[0][0]    │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 32, 32,    │    295,168 │ max_pooling2d_3[… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_4     │ (None, 16, 16,    │          0 │ conv2d_4[0][0]    │
│ (MaxPooling2D)      │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 16, 16,    │  1,180,160 │ max_pooling2d_4[… │
│                     │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_transpose    │ (None, 32, 32,    │  1,179,904 │ conv2d_5[0][0]    │
│ (Conv2DTranspose)   │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 32, 32,    │          0 │ conv2d_4[0][0],   │
│ (Concatenate)       │ 512)              │            │ conv2d_transpose… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 32, 32,    │  1,179,904 │ concatenate_1[0]… │
│                     │ 256)              │            │                 

 Total params: 4,662,790 (17.79 MB)

 Trainable params: 4,662,790 (17.79 MB)

 Non-trainable params: 0 (0.00 B)

In [22]:
class FullVideoGenModel(tf.keras.Model):
    def __init__(self, encoder, decoder, diffusion_model, num_frames=32):
        super(FullVideoGenModel, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.diffusion = diffusion_model
        self.num_frames = num_frames

    def call(self, inputs, training=False):
        video_input, text_embedding, timesteps = inputs

        # Encode video + text to latent
        z = self.encoder([video_input, text_embedding])  # shape: (B, latent_dim)
        
        # Decode latent into video frames
        reconstructed_frames = self.decoder(z)  # shape: (B, 32, H, W, 3)

        # Apply diffusion model on each frame
        diffused_frames = []
        for t in range(self.num_frames):
            frame = reconstructed_frames[:, t, :, :, :]     # (B, H, W, 3)
            timestep = tf.expand_dims(timesteps[:, t], axis=-1)  # shape: (B, 1)  # shape: (B, 1)
            diffused = self.diffusion([frame, timestep])      # call DiffusionUNet
            diffused_frames.append(diffused)                # (B, H, W, 3)

        # Stack frames back into video shape
        final_video = tf.stack(diffused_frames, axis=1)  # (B, 32, H, W, 3)
        return final_video


In [23]:
import tensorflow as tf
import os
from datetime import datetime

In [24]:
def create_tf_dataset(video_data, text_embeddings, batch_size=4):
    ground_truth = video_data  # Target is original video (denoised)
    dataset = tf.data.Dataset.from_tensor_slices((video_data, text_embeddings, ground_truth))
    dataset = dataset.shuffle(buffer_size=100).batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return dataset


In [25]:
# Optimizer and Loss
learning_rate = 1e-3
optimizer = tf.keras.optimizers.Adam(learning_rate)

# Mean Squared Error for pixel-level accuracy
loss_fn = tf.keras.losses.MeanAbsoluteError()


In [26]:
import matplotlib.pyplot as plt

def generate_and_show_frames(model, dataset, num_samples=1):
    video_input, text_embedding, ground_truth = next(iter(dataset))
    batch_size = video_input.shape[0]
    timesteps = tf.random.uniform((batch_size, 32), minval=0, maxval=1000, dtype=tf.int32)

    predictions = model((video_input, text_embedding, timesteps), training=False)

    for i in range(num_samples):
        fig, axes = plt.subplots(1, 5, figsize=(15, 3))
        fig.suptitle(f"Sample {i+1}: Generated Frames")

        for j in range(5):  # show 5 frames from the video
            axes[j].imshow(predictions[i, j])
            axes[j].axis('off')
        plt.show()


In [33]:
import time

In [46]:
def train_model(model, dataset, epochs=100):
    for epoch in range(epochs):
        start_time = time.time()
        print(f"\nEpoch {epoch + 1}/{epochs}")
        epoch_loss = []

        for batch, (video_input, text_embedding, ground_truth) in enumerate(dataset):
            batch_size = video_input.shape[0]
            timesteps = tf.random.uniform((batch_size, 32), minval=0, maxval=1000, dtype=tf.int32)

            with tf.GradientTape() as tape:
                predictions = model((video_input, text_embedding, timesteps), training=True)
                loss = loss_fn(ground_truth, predictions)

            grads = tape.gradient(loss, model.trainable_variables)
            optimizer.apply_gradients(zip(grads, model.trainable_variables))
            epoch_loss.append(loss.numpy())

            if batch % 10 == 0:
                print(f"  Batch {batch}: Loss = {loss.numpy():.4f}")

        mean_loss = np.mean(epoch_loss)
        epoch_time = time.time() - start_time
        print(f">>> Epoch {epoch + 1} Mean Loss: {mean_loss:.4f}")
        print(f" Time taken for epoch {epoch + 1}: {epoch_time:.2f} seconds")

        # Save weights every 5 epochs
        try:
            if (epoch + 1) % 5 == 0:
                model.save_weights(f"model_epoch_{epoch+1}.h5")

                # Generate and show frames from the model
                print("Generating and showing sample frames from the model...")
                generate_and_show_frames(model, dataset)
        except:
            print(f"not saved{epoch+1}")

In [47]:
model = FullVideoGenModel(encoder, decoder, diffusion_model)

In [48]:
model.summary()

Model: "full_video_gen_model_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ video_encoder (Functional)      │ (None, 512)            │   239,679,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ video_decoder (Functional)      │ (None, 32, 128, 128,   │    27,314,176 │
│                                 │ 3)                     │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ DiffusionUNet (Functional)      │ (None, 128, 128, 3)    │     4,662,790 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 271,656,838 (1.01 GB)

 Trainable params: 271,656,838 (1.01 GB)

 Non-trainable params: 0 (0.00 B)

In [49]:
dataset = create_tf_dataset(video_data, text_embeddings, batch_size=4)

In [50]:
dataset

<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 32, 128, 128, 3), dtype=tf.float64, name=None), TensorSpec(shape=(None, 512), dtype=tf.float32, name=None), TensorSpec(shape=(None, 32, 128, 128, 3), dtype=tf.float64, name=None))>

In [51]:
train_model(model, dataset)


Epoch 1/100
  Batch 0: Loss = 0.2246
  Batch 10: Loss = 0.2226
  Batch 20: Loss = 0.2269
  Batch 30: Loss = 0.2218
  Batch 40: Loss = 0.2707


2025-04-05 08:31:25.399091: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


>>> Epoch 1 Mean Loss: 0.2400
 Time taken for epoch 1: 2232.62 seconds

Epoch 2/100
  Batch 0: Loss = 0.2647
  Batch 10: Loss = 0.2828
  Batch 20: Loss = 0.2161
  Batch 30: Loss = 0.2418
  Batch 40: Loss = 0.1966
>>> Epoch 2 Mean Loss: 0.2392
 Time taken for epoch 2: 2229.72 seconds

Epoch 3/100
  Batch 0: Loss = 0.2287
  Batch 10: Loss = 0.2286
  Batch 20: Loss = 0.2288
  Batch 30: Loss = 0.1824
  Batch 40: Loss = 0.1984


2025-04-05 09:45:42.800680: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


>>> Epoch 3 Mean Loss: 0.2284
 Time taken for epoch 3: 2227.68 seconds

Epoch 4/100
  Batch 0: Loss = 0.2447
  Batch 10: Loss = 0.2415
  Batch 20: Loss = 0.2457
  Batch 30: Loss = 0.2329
  Batch 40: Loss = 0.1925
>>> Epoch 4 Mean Loss: 0.2068
 Time taken for epoch 4: 2235.09 seconds

Epoch 5/100
  Batch 0: Loss = 0.1675
  Batch 10: Loss = 0.1976
  Batch 20: Loss = 0.2050
  Batch 30: Loss = 0.1769
  Batch 40: Loss = 0.2003
>>> Epoch 5 Mean Loss: 0.2110
 Time taken for epoch 5: 2233.49 seconds
not saved5

Epoch 6/100
  Batch 0: Loss = 0.2456
  Batch 10: Loss = 0.2515
  Batch 20: Loss = 0.1455
  Batch 30: Loss = 0.2319
  Batch 40: Loss = 0.1634
>>> Epoch 6 Mean Loss: 0.2046
 Time taken for epoch 6: 2243.32 seconds

Epoch 7/100
  Batch 0: Loss = 0.1987
  Batch 10: Loss = 0.2143
  Batch 20: Loss = 0.2392
  Batch 30: Loss = 0.1906
  Batch 40: Loss = 0.2009


...

>>> Epoch 15 Mean Loss: 0.1912
 Time taken for epoch 15: 2238.86 seconds
not saved15

Epoch 16/100
  Batch 0: Loss = 0.1767
  Batch 10: Loss = 0.1804
  Batch 20: Loss = 0.1830
  Batch 30: Loss = 0.1728
  Batch 40: Loss = 0.2115
>>> Epoch 16 Mean Loss: 0.1891
 Time taken for epoch 16: 2234.61 seconds

Epoch 17/100
  Batch 0: Loss = 0.2015
  Batch 10: Loss = 0.2040
  Batch 20: Loss = 0.1699
  Batch 30: Loss = 0.2083
  Batch 40: Loss = 0.1609
>>> Epoch 17 Mean Loss: 0.1890
 Time taken for epoch 17: 2234.05 seconds

Epoch 18/100
  Batch 0: Loss = 0.2224
  Batch 10: Loss = 0.2034
  Batch 20: Loss = 0.1735
  Batch 30: Loss = 0.2293
  Batch 40: Loss = 0.1678
>>> Epoch 18 Mean Loss: 0.1878
 Time taken for epoch 18: 2234.19 seconds

Epoch 19/100
  Batch 0: Loss = 0.1572
  Batch 10: Loss = 0.2008
  Batch 20: Loss = 0.2033
  Batch 30: Loss = 0.1857
  Batch 40: Loss = 0.1212
>>> Epoch 19 Mean Loss: 0.1895
 Time taken for epoch 19: 2235.17 seconds

Epoch 20/100
  Batch 0: Loss = 0.1701
  Batch 10: 

2025-04-06 03:09:54.245362: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


>>> Epoch 31 Mean Loss: 0.1722
 Time taken for epoch 31: 2230.22 seconds

Epoch 32/100
  Batch 0: Loss = 0.1696
  Batch 10: Loss = 0.1608
  Batch 20: Loss = 0.1747
  Batch 30: Loss = 0.2016
  Batch 40: Loss = 0.1160
>>> Epoch 32 Mean Loss: 0.1674
 Time taken for epoch 32: 2229.26 seconds

Epoch 33/100
  Batch 0: Loss = 0.2091
  Batch 10: Loss = 0.1705
  Batch 20: Loss = 0.2003
  Batch 30: Loss = 0.1402
  Batch 40: Loss = 0.1769
>>> Epoch 33 Mean Loss: 0.1662
 Time taken for epoch 33: 2229.75 seconds

Epoch 34/100
  Batch 0: Loss = 0.1629
  Batch 10: Loss = 0.1979
  Batch 20: Loss = 0.1586
  Batch 30: Loss = 0.1314
  Batch 40: Loss = 0.1602
>>> Epoch 34 Mean Loss: 0.1642
 Time taken for epoch 34: 2230.22 seconds

Epoch 35/100
  Batch 0: Loss = 0.1611
  Batch 10: Loss = 0.1944
  Batch 20: Loss = 0.1990
  Batch 30: Loss = 0.1825
  Batch 40: Loss = 0.1779
>>> Epoch 35 Mean Loss: 0.1624
 Time taken for epoch 35: 2224.89 seconds
not saved35

Epoch 36/100
  Batch 0: Loss = 0.1838
  Batch 10: 

2025-04-06 22:57:02.256835: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


>>> Epoch 63 Mean Loss: 0.1352
 Time taken for epoch 63: 2222.70 seconds

Epoch 64/100
  Batch 0: Loss = 0.1147
  Batch 10: Loss = 0.1067
  Batch 20: Loss = 0.1093
  Batch 30: Loss = 0.1147
  Batch 40: Loss = 0.1477
>>> Epoch 64 Mean Loss: 0.1360
 Time taken for epoch 64: 2222.43 seconds

Epoch 65/100
  Batch 0: Loss = 0.1266
  Batch 10: Loss = 0.1171
  Batch 20: Loss = 0.1502
  Batch 30: Loss = 0.1223
  Batch 40: Loss = 0.1004
>>> Epoch 65 Mean Loss: 0.1343
 Time taken for epoch 65: 2222.40 seconds
not saved65

Epoch 66/100
  Batch 0: Loss = 0.1345
  Batch 10: Loss = 0.0992
  Batch 20: Loss = 0.1579
  Batch 30: Loss = 0.1496
  Batch 40: Loss = 0.1326
>>> Epoch 66 Mean Loss: 0.1332
 Time taken for epoch 66: 2222.82 seconds

Epoch 67/100
  Batch 0: Loss = 0.1126
  Batch 10: Loss = 0.1389
  Batch 20: Loss = 0.1126
  Batch 30: Loss = 0.1205
  Batch 40: Loss = 0.1161
>>> Epoch 67 Mean Loss: 0.1317
 Time taken for epoch 67: 2221.64 seconds

Epoch 68/100
  Batch 0: Loss = 0.0968
  Batch 10: 

In [54]:
model.save_weights(f"EncoderDecoder_Diffusion_final.weights.h5")